# Lab 1: Tracing Basics & Types of Runs

## Difficulty: Beginner | ~30 min | No prerequisites

Learn how to instrument your code with LangSmith's `@traceable` decorator, understand the anatomy of a trace, and recognize the different run types LangSmith captures.

In [ ]:
!pip install langsmith>=0.1.0 openai>=1.0.0 python-dotenv>=1.0.0

This installs the exact versions of every library used in this lab.

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from .env file
# This reads OPENROUTER_API_KEY, LANGSMITH_API_KEY, LANGSMITH_TRACING, and LANGSMITH_PROJECT
load_dotenv()

# Verify all required keys are present
# If any are missing, this will raise an AssertionError with a helpful message
assert os.getenv("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY in .env"
assert os.getenv("LANGSMITH_API_KEY"), "Missing LANGSMITH_API_KEY in .env"
assert os.getenv("LANGSMITH_TRACING") == "true", "Missing LANGSMITH_TRACING=true in .env"

print("✓ All environment variables loaded successfully")

This loads your API keys from the `.env` file and verifies they're present. The `LANGSMITH_TRACING=true` variable tells the SDK to send traces to LangSmith.

In [ ]:
from langsmith import traceable, Client

# Initialize the LangSmith client
# This client connects to LangSmith's servers and sends traces automatically
ls_client = Client()

# Verify the client connected successfully
print(f"✓ LangSmith client initialized")
print(f"  Project: {os.getenv('LANGSMITH_PROJECT')}")
print(f"  Tracing: {os.getenv('LANGSMITH_TRACING')}")

The LangSmith client connects to LangSmith's servers. This is what sends your traces automatically when you use `@traceable`.

In [ ]:
from openai import OpenAI

# Initialize OpenAI client pointing to OpenRouter
# OpenRouter is OpenAI-compatible, so we just change the base_url
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

print("✓ OpenRouter client initialized (via OpenAI SDK)")

We use OpenAI's SDK but point it to OpenRouter's endpoint. OpenRouter is OpenAI-compatible, so the same client works — we just change the `base_url` and `api_key`.

In [ ]:
@traceable(run_type="llm", name="simple_chat_completion")
def simple_chat(user_message: str) -> str:
    """Call chat model via OpenRouter with a single message."""
    # Send the message to a free model on OpenRouter
    response = openai_client.chat.completions.create(
        model="deepseek/deepseek-chat-v3-0324:free",  # Free, fast, reliable
        messages=[{"role": "user", "content": user_message}]
    )
    # Return just the text content from the response
    return response.choices[0].message.content

This function calls a chat model via OpenRouter. The `@traceable` decorator records the input (`user_message`) and output (the response). We set `run_type="llm"` to mark this as an LLM operation.

In [ ]:
# Call the traced function
response = simple_chat("What is 2+2? Reply with just the number.")

print(f"Response: {response}")

When you run this cell, LangSmith records a trace with a single `llm` run. The trace includes the input message, the response, latency, and token counts.

In [ ]:
# List recent traces to verify
traces = list(ls_client.list_runs(
    project_name=os.getenv("LANGSMITH_PROJECT"),
    is_root=True,  # Only top-level traces
    limit=5
))

print(f"✓ Found {len(traces)} recent trace(s)")
for trace in traces:
    print(f"  - {trace.name} ({trace.run_type}) - {trace.status}")

This queries LangSmith for your recent traces. You should see your `simple_chat_completion` run with `run_type="llm"` and `status="success"`.

In [ ]:
@traceable(run_type="chain", name="question_answering_chain")
def question_answerer(question: str) -> str:
    """A simple chain that wraps an LLM call."""
    # Add system context to the question
    prompt = f"Answer this question concisely: {question}"
    
    # Call the LLM (this will be a child run of the chain)
    answer = simple_chat(prompt)
    
    return answer

This creates a **chain run** that wraps the LLM call. When you run this, you'll see a parent `chain` run with a child `llm` run inside it — this is the run tree hierarchy in action.

In [ ]:
# Call the wrapper chain
response = question_answerer("What is the capital of France?")

print(f"Response: {response}")

When you run this, LangSmith records a trace with a `chain` run containing an `llm` child run. Check the LangSmith UI to see the nested structure.

In [ ]:
# List all traces to see both calls
traces = list(ls_client.list_runs(
    project_name=os.getenv("LANGSMITH_PROJECT"),
    is_root=True,
    limit=10
))

print(f"✓ Found {len(traces)} trace(s)")
for i, trace in enumerate(traces, 1):
    print(f"\n  Trace {i}: {trace.name}")
    print(f"    Type: {trace.run_type}")
    print(f"    Status: {trace.status}")
    print(f"    Latency: {trace.total_tokens or 'N/A'} tokens")

This shows all your traces side by side. The first has a single `llm` run; the second has a `chain` parent with `llm` child. This is exactly the baseline trace you'll compare against in future labs.